In [ ]:

import torch
import torch.nn as nn
from torch.nn import functional as F

# ==========================================
# 1. CONFIGURATION & HYPERPARAMETERS
# ==========================================
batch_size = 32          # Number of independent sequences processed in parallel
block_size = 256         # Maximum context length (window size)
max_iters = 5000         # Total training iterations
eval_interval = 500      # Intentional interval to estimate loss
learning_rate = 3e-4     # Adam learning rate
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 384             # Embedding dimension size
n_head = 6               # Number of attention heads (384 / 6 = 64 dimension per head)
n_layer = 6              # Number of transformer blocks stacked
dropout = 0.2            # Dropout probability

torch.manual_seed(1337)

# ==========================================
# 2. CAUSAL MULTI-HEAD SELF-ATTENTION
# ==========================================
class Head(nn.Module):
    """ Single head of causal self-attention """
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        # Register a lower-triangular causal mask buffer (not a trainable parameter)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)   # (B, T, head_size)
        q = self.query(x) # (B, T, head_size)

        # Compute attention scores ("affinities") scaled by the square root of head size
        wei = q @ k.transpose(-2, -1) * (C ** -0.5) # (B, T, head_size) @ (B, head_size, T) -> (B, T, T)
        # Mask future tokens to prevent the model from looking ahead
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        # Perform the weighted aggregation of values
        v = self.value(x) # (B, T, head_size)
        out = wei @ v     # (B, T, T) @ (B, T, head_size) -> (B, T, head_size)
        return out

class MultiHeadAttention(nn.Module):
    """ Multiple heads of self-attention running in parallel """
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd) # Projection layer back into residual pathway
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # Concatenate outputs from all heads along the channel dimension
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

# ==========================================
# 3. POSITION-WISE FEED-FORWARD NETWORK
# ==========================================
class FeedForward(nn.Module):
    """ A simple linear layer followed by a non-linearity (GELU) """
    def __init__(self, n_embd):
        super().__init__()
        # Standard Transformer architecture expands hidden dimension by a factor of 4
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

# ==========================================
# 4. TRANSFORMER BLOCK (LAYER)
# ==========================================
class Block(nn.Module):
    """ Transformer block: communicates (attention) then computes (feedforward) """
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        # Pre-Layer Normalization architecture with residual skip-connections
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

# ==========================================
# 5. CORE LLM DECODER ARCHITECTURE
# ==========================================
class MiniLanguageModel(nn.Module):
    """ Complete Decoder-Only Transformer Language Model """
    def __init__(self, vocab_size):
        super().__init__()
        # Each token looks up its dense vector embedding
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        # Each position looks up its structural location embedding
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        # Stack sequential transformer layers
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        # Final layer normalization
        self.ln_f = nn.LayerNorm(n_embd)
        # Language modeling head mapping hidden state back to vocabulary logits
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # Retrieve structural embeddings
        tok_emb = self.token_embedding_table(idx) # (B, T, n_embd)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T, n_embd)
        x = tok_emb + pos_emb # Combine content and spatial location (B, T, n_embd)

        # Pass through the core network backbone
        x = self.blocks(x) # (B, T, n_embd)
        x = self.ln_f(x)   # (B, T, n_embd)
        logits = self.lm_head(x) # (B, T, vocab_size)

        loss = None
        if targets is not None:
            # Flatten cross-entropy inputs to evaluate across the sequence
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        """ Generate novel text auto-regressively given a starting context """
        for _ in range(max_new_tokens):
            # Crop current context if it exceeds the maximum architectural block size
            idx_cond = idx[:, -block_size:]
            # Get next-step predictions
            logits, loss = self(idx_cond)
            # Focus strictly on the final index step to make the next prediction
            logits = logits[:, -1, :] # Becomes (B, C)
            # Convert predictions into probability distributions
            probs = F.softmax(logits, dim=-1) # (B, C)
            # Sample next item from the generated categorical distributions
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # Concat sampled token to ongoing history context
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

In [ ]:
import os
import urllib.request

# ==========================================
# 6. DATASET DOWNLOADING & TOKENIZATION
# ==========================================
# Download a tiny text corpus for training if it doesn't exist locally
data_url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
data_path = "input.txt"

if not os.path.exists(data_path):
    print("Downloading training dataset...")
    urllib.request.urlretrieve(data_url, data_path)

with open(data_path, 'r', encoding='utf-8') as f:
    text = f.read()

# Determine vocabulary characteristics
chars = sorted(list(set(text)))
vocab_size = len(chars)

# Build a fast character-to-integer encoder and decoder mapping
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]          # String encoder
decode = lambda l: ''.join([itos[i] for i in l])  # List of integers decoder

# Split dataset into training (90%) and validation (10%) arrays
data = torch.tensor(encode(text), dtype=torch.long)
n_train = int(0.9 * len(data))
train_data = data[:n_train]
val_data = data[n_train:]

def get_batch(split):
    """ Generate a small batch of data containing inputs X and targets Y """
    data_split = train_data if split == 'train' else val_data
    # Generate random starting index positions within the dataset
    ix = torch.randint(len(data_split) - block_size, (batch_size,))
    x = torch.stack([data_split[i:i+block_size] for i in ix])
    # Targets are offset by one position forward in time
    y = torch.stack([data_split[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss(model):
    """ Periodically calculate evaluation losses to monitor overfitting """
    out = {}
    model.eval() # Disable dropout during loss metrics estimation
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train() # Reactivate dropout training configurations
    return out

In [ ]:
# ==========================================
# 7. TRAINING LOOP EXECUTION
# ==========================================
print(device)
model = MiniLanguageModel(vocab_size)
model = model.to(device)

# Initialize standard PyTorch AdamW optimization engine
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

print(f"Training initiated. Total iterations: {max_iters}...")
for iter in range(max_iters):

    # Periodically evaluate current loss on train and validation sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss(model)
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # Sample a batch of inputs and expected outputs
    xb, yb = get_batch('train')

    # Evaluate the loss through a standard forward pass
    logits, loss = model(xb, yb)

    # Zero out prior gradient buffers, run backward tracking, and optimize weights
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# ==========================================
# 8. SAVING THE TRAINED MODEL CHECKPOINT
# ==========================================
model_filename = "mini_llm_model.pt"
checkpoint = {
    'model_state_dict': model.state_dict(),
    'vocab_size': vocab_size,
    'chars': chars
}
torch.save(checkpoint, model_filename)
print(f"Model optimization finalized. Weights stored as '{model_filename}' successfully.")

# ==========================================
# 9. TESTING THE SAVED CHECKPOINT (INFERENCE)
# ==========================================
print("\n--- Testing Model Generation Output ---")
# Prompt context: pass a single starting token (index zero or newline) to generate from
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_tokens = model.generate(context, max_new_tokens=200)[0].tolist()
print(decode(generated_tokens))

cuda
Training initiated. Total iterations: 5000...
step 0: train loss 4.2890, val loss 4.2861
step 500: train loss 2.1037, val loss 2.1617
step 1000: train loss 1.6901, val loss 1.8476
step 1500: train loss 1.5091, val loss 1.6977
step 2000: train loss 1.4014, val loss 1.6145
step 2500: train loss 1.3270, val loss 1.5621
step 3000: train loss 1.2815, val loss 1.5235
step 3500: train loss 1.2343, val loss 1.5097
step 4000: train loss 1.1998, val loss 1.4902
step 4500: train loss 1.1656, val loss 1.4836
step 4999: train loss 1.1352, val loss 1.4863
Model optimization finalized. Weights stored as 'mini_llm_model.pt' successfully.

--- Testing Model Generation Output ---

AUFINGARENE:
Sometimes your face.

NORTHUMBERLAND:
My looks:
No soft warrant is, 'twixt commund:
As my promison, on mortation the wisdom war
To visitor us, I send under them at it the souls

ISABELLA:


In [ ]:
# ==========================================
# 10. RELOAD ARCHITECTURE FOR INFERENCE
# ==========================================
def load_and_generate(text_prompt, generation_length=150):
    # Reload stored checkpoint items
    checkpoint = torch.load("mini_llm_model.pt", map_location=device)

    # Reconstruct exact tokenizer conditions from checkpoint metrics
    saved_chars = checkpoint['chars']
    saved_vocab_size = checkpoint['vocab_size']
    s_toi = { ch:i for i,ch in enumerate(saved_chars) }
    s_itos = { i:ch for i,ch in enumerate(saved_chars) }

    # Instantiate raw architecture structure and push state dictionaries
    loaded_model = MiniLanguageModel(saved_vocab_size).to(device)
    loaded_model.load_state_dict(checkpoint['model_state_dict'])
    loaded_model.eval() # Configure model properties strictly for generation evaluation

    # Encode prompt string data into numeric values
    encoded_prompt = [s_toi[c] for c in text_prompt if c in s_toi]
    input_tensor = torch.tensor([encoded_prompt], dtype=torch.long, device=device)

    # Run generative loop prediction tasks
    output_indices = loaded_model.generate(input_tensor, max_new_tokens=generation_length)[0].tolist()

    # Decode integers back to text
    return ''.join([s_itos[i] for i in output_indices])

# Example usage:
print(load_and_generate("ROMEO: hii my name is ", generation_length=100))

ROMEO: hii my name is this wounded sink; for,
there is no lesser such a ghost advantagement
provost you: this is in mortal


In [ ]:
import os
import torch

# Assuming all your prior classes (Head, MultiHeadAttention, FeedForward, Block, MiniLanguageModel)
# and data variables (vocab_size, train_data, val_data, get_batch, estimate_loss, device, etc.) are present above.

# Checkpoint filename setup - using the exact file you have
model_filename = "mini_llm_model.pt"

# Initialize model and optimizer structure
model = MiniLanguageModel(vocab_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# Tracking state defaults
start_iter = 0

# ==========================================
# 1. LOAD MODEL STATE WITH SAFEST FALLBACKS
# ==========================================
if os.path.exists(model_filename):
    print(f"Found existing checkpoint '{model_filename}'. Resuming training...")

    # Load checkpoint dictionary safely across active compute devices
    checkpoint = torch.load(model_filename, map_location=device)

    # 1. Always load the model weights (this prevents losing progress)
    model.load_state_dict(checkpoint['model_state_dict'])
    print("-> Model weights loaded successfully.")

    # 2. Safely check for the optimizer state
    if 'optimizer_state_dict' in checkpoint:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        print("-> Optimizer state found and loaded.")
    else:
        print("-> Warning: 'optimizer_state_dict' not found in file. Initializing a fresh optimizer on top of current weights.")

    # 3. Safely check for the iteration count
    if 'iteration' in checkpoint:
        start_iter = checkpoint['iteration'] + 1
        print(f"-> Resuming from step {start_iter}.")
    else:
        print("-> Warning: 'iteration' tracking data missing. Defaulting back to step 0 (weights preserved).")
        start_iter = 0

else:
    print("No previous checkpoint detected. Starting training from scratch (step 0)...")




# ==========================================
# 2. RESUMED TRAINING LOOP
# ==========================================
# Run training from our starting step up to the target max_iters threshold
for iter in range(start_iter, max_iters):

    # Periodically estimate evaluation loss
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss(model)
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # Sample batch sequences forward
    xb, yb = get_batch('train')

    # Forward loss processing and backward gradient application
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    # OPTIONAL: Mid-train safety saving checkpoint every eval_interval steps
    if iter % eval_interval == 0 and iter > start_iter:
        print(f"Saving temporary progress checkpoint at step {iter}...")
        checkpoint = {
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(), # Critical for internal momentum
            'iteration': iter,                              # Tracks structural index boundary [1]
            'vocab_size': vocab_size,
            'chars': chars
        }
        torch.save(checkpoint, model_filename)


# ==========================================
# 3. FINAL RESAVING LOGIC
# ==========================================
print("\nTraining run phase complete. Storing updated progress...")
final_checkpoint = {
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'iteration': max_iters - 1, # Marks total iteration limit reached
    'vocab_size': vocab_size,
    'chars': chars
}
torch.save(final_checkpoint, model_filename)
print(f"State configuration metrics updated securely inside '{model_filename}'.")

Found existing checkpoint 'mini_llm_model.pt'. Resuming training...
-> Model weights loaded successfully.
-> Optimizer state found and loaded.
-> Resuming from step 501.
step 1000: train loss 1.0788, val loss 1.4801
Saving temporary progress checkpoint at step 1000...
step 1500: train loss 1.0518, val loss 1.4914
Saving temporary progress checkpoint at step 1500...


KeyboardInterrupt: 

In [ ]:
import os
import torch
import torch.nn as nn
from torch.nn import functional as F

# ===================================================
# 1. SETUP CORE CONFIGURATIONS
# ===================================================
device = 'cuda' if torch.cuda.is_available() else 'cpu'
block_size = 256
n_embd = 384
dropout = 0.2
n_head = 6
n_layer = 6

# Existing checkpoint file path
model_filename = "mini_llm_model.pt"

# Define the explicit vocabulary characters from your pretraining setup
base_chars = sorted(list(set("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789 .,\n;:!?'-()")))
special_tokens = ['[PAD]', '[EOS]']
all_tokens = base_chars + special_tokens

# Map dictionaries
stoi = {ch: i for i, ch in enumerate(all_tokens)}
itos = {i: ch for i, ch in enumerate(all_tokens)}

PAD_ID = stoi['[PAD]']
EOS_ID = stoi['[EOS]']

# ===================================================
# 2. INSTRUCTION DATA PREPARATION WITH [PAD]
# ===================================================
finetuning_data = [
    {"prompt": "Q: What is 2+2?\nA:", "completion": "4.[EOS]"},
    {"prompt": "Q: Who wrote Hamlet?\nA:", "completion": "Shakespeare.[EOS]"},
    {"prompt": "Q: Capital of France?\nA:", "completion": "Paris.[EOS]"},
    {"prompt": "Q: Is fire hot or cold?\nA:", "completion": "Fire is hot.[EOS]"}
]

def tokenize_and_pad(dataset, max_len=64):
    all_input_ids = []
    all_targets = []
    for item in dataset:
        p_ids = [stoi[c] for c in item['prompt'] if c in stoi]
        c_ids = [stoi[c] for c in item['completion'] if c in stoi]

        input_sequence = p_ids + c_ids
        target_sequence = ([-100] * len(p_ids)) + c_ids  # Only evaluate completions, ignore prompt tokens (-100)

        if len(input_sequence) > max_len:
            input_sequence = input_sequence[:max_len]
            target_sequence = target_sequence[:max_len]

        pad_len = max_len - len(input_sequence)
        input_sequence += [PAD_ID] * pad_len
        target_sequence += [-100] * pad_len  # Pad tokens ignored by loss function (-100)

        all_input_ids.append(torch.tensor(input_sequence))
        all_targets.append(torch.tensor(target_sequence))
    return torch.stack(all_input_ids), torch.stack(all_targets)

X_train, Y_train = tokenize_and_pad(finetuning_data, max_len=64)

# ===================================================
# 3. LOAD EXISTING MODEL & STRUCTURAL RESIZING
# ===================================================
# Initialize base model architecture matching the original vocabulary length
model = MiniLanguageModel(vocab_size=len(base_chars)).to(device)

if os.path.exists(model_filename):
    print(f"Found existing checkpoint '{model_filename}'. Resuming model state...")
    checkpoint = torch.load(model_filename, map_location=device)

    # Peek at saved weights matrix dimension to verify its size
    saved_weight_shape = checkpoint['model_state_dict']['token_embedding_table.weight'].shape[0]

    if saved_weight_shape == len(base_chars):
        print("-> Detected base model checkpoint. Loading state and expanding embeddings for special tokens...")
        model.load_state_dict(checkpoint['model_state_dict'])
        model.resize_token_embeddings(len(all_tokens))
    elif saved_weight_shape == len(all_tokens):
        print("-> Detected already resized model checkpoint. Initializing resized layers before weights load...")
        model.resize_token_embeddings(len(all_tokens))
        model.load_state_dict(checkpoint['model_state_dict'])
    else:
        raise ValueError(f"Checkpoint vocabulary dimension mismatch ({saved_weight_shape}). Cannot reliably adapt structure.")
else:
    print("Warning: No previous checkpoint detected! Initializing empty model from scratch...")
    model.resize_token_embeddings(len(all_tokens))

# ===================================================
# 4. FINE-TUNING EXECUTION LOOP
# ===================================================
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5) # Fine-tuning utilizes a lower learning rate
epochs = 100
batch_size_ft = 2

print("\nBeginning instruction fine-tuning phase...")
model.train()

for epoch in range(epochs):
    permutation = torch.randperm(X_train.size(0))
    epoch_loss = 0

    for i in range(0, X_train.size(0), batch_size_ft):
        optimizer.zero_grad(set_to_none=True)

        indices = permutation[i:i+batch_size_ft]
        batch_x, batch_y = X_train[indices].to(device), Y_train[indices].to(device)

        logits, loss = model(batch_x, batch_y)

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1}/{epochs} | Fine-Tuning Loss: {epoch_loss / (X_train.size(0)/batch_size_ft):.4f}")

# Save updated fine-tuned model checkpoint
print("\nSaving updated fine-tuned progress checkpoint...")
fine_tuned_checkpoint = {
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'vocab_size': len(all_tokens),
    'chars': all_tokens
}
torch.save(fine_tuned_checkpoint, model_filename)
print(f"Fine-tuned state safely exported to '{model_filename}'.")

# ===================================================
# 5. TESTING THE MODEL GENERATION OUTPUT
# ===================================================
model.eval()
print("\n--- Testing Fine-Tuned Model Inference ---")

def ask_model(question_prompt):
    encoded_query = [stoi[c] for c in question_prompt if c in stoi]
    input_tensor = torch.tensor([encoded_query], dtype=torch.long, device=device)

    output_ids = model.generate(input_tensor, max_new_tokens=20).tolist()

    # Filter out internal [PAD] structures during conversational decode transitions
    decoded_string = "".join([itos[i] for i in output_ids[0] if i != PAD_ID])
    return decoded_string

print(ask_model("Q: What is 2+2?\nA:"))
print(ask_model("Q: Capital of France?\nA:"))

Found existing checkpoint 'mini_llm_model.pt'. Resuming model state...


ValueError: Checkpoint vocabulary dimension mismatch (65). Cannot reliably adapt structure.

In [ ]:
import os
import torch
import torch.nn as nn
from torch.nn import functional as F

# ===================================================
# 1. SETUP CORE CONFIGURATIONS
# ===================================================
device = 'cuda' if torch.cuda.is_available() else 'cpu'
block_size = 256
n_embd = 384
dropout = 0.2
n_head = 6
n_layer = 6

# Existing checkpoint file path
model_filename = "mini_llm_model.pt"

# 1. Define the original pretrained character list
base_chars = sorted(list(set("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789 .,\n;:!?'-()")))

# 2. Define the complete production-grade special token suite
special_tokens = ['[PAD]', '[BOS]', '[EOS]', '[SYSTEM]', '[USER]', '[ASSISTANT]', '[THOUGHT]']
all_tokens = base_chars + special_tokens

# 3. Create absolute vocabulary maps
stoi = {ch: i for i, ch in enumerate(all_tokens)}
itos = {i: ch for i, ch in enumerate(all_tokens)}

# Quick structural ID references
PAD_ID = stoi['[PAD]']
BOS_ID = stoi['[BOS]']
EOS_ID = stoi['[EOS]']

# ===================================================
# 2. SYSTEM INSTRUCTION CHAT DATA SAMPLES
# ===================================================
# Multi-turn system data with explicit logic branches
chat_dataset = [
    {
        "system": "You are a math helper. Answer concisely.",
        "user": "What is 2+2?",
        "thought": "The user is asking a basic arithmetic addition query. 2 + 2 equals 4.",
        "assistant": "2 + 2 is 4."
    },
    {
        "system": "You are an expert geographer.",
        "user": "What is the capital of France?",
        "thought": "The question is about country capitals. France's capital city is Paris.",
        "assistant": "The capital of France is Paris."
    }
]

def format_and_tokenize(dataset, max_len=128):
    """
    Parses structural dialogues into a clean sequential tensor string.
    Masks prompts, system instructions, and internal thoughts with -100
    so the model is only optimized to predict the assistant's response.
    """
    all_input_ids = []
    all_targets = []

    for item in dataset:
        # Build tokenized sub-blocks
        sys_tokens  = [stoi[c] for c in f"[SYSTEM]{item['system']}" if c in stoi]
        user_tokens = [stoi[c] for c in f"[USER]{item['user']}" if c in stoi]
        thgt_tokens = [stoi[c] for c in f"[THOUGHT]{item['thought']}" if c in stoi]
        asst_tokens = [stoi[c] for c in f"[ASSISTANT]{item['assistant']}[EOS]" if c in stoi]

        # Combine everything starting with the Beginning of Sequence token
        input_sequence = [BOS_ID] + sys_tokens + user_tokens + thgt_tokens + asst_tokens

        # Mask everything except assistant responses out of training gradient evaluations via -100
        non_asst_len = 1 + len(sys_tokens) + len(user_tokens) + len(thgt_tokens)
        target_sequence = ([-100] * non_asst_len) + asst_tokens

        # Truncate sequences if they overshoot our operational context blocks
        if len(input_sequence) > max_len:
            input_sequence = input_sequence[:max_len]
            target_sequence = target_sequence[:max_len]

        # Apply padding dynamically to build clean tensors
        pad_len = max_len - len(input_sequence)
        input_sequence += [PAD_ID] * pad_len
        target_sequence += [-100] * pad_len

        all_input_ids.append(torch.tensor(input_sequence))
        all_targets.append(torch.tensor(target_sequence))

    return torch.stack(all_input_ids), torch.stack(all_targets)

X_train, Y_train = format_and_tokenize(chat_dataset, max_len=128)

# ===================================================
# 3. SECURE ADAPTIVE CHECKPOINT LOADING
# ===================================================
# Instantiate standard model layout on base configurations first
model = MiniLanguageModel(vocab_size=len(base_chars)).to(device)

if os.path.exists(model_filename):
    print(f"Found existing checkpoint '{model_filename}'. Validating structures...")
    checkpoint = torch.load(model_filename, map_location=device)

    # Analyze raw weight shape inside the checkpoint file
    saved_vocab_size = checkpoint['model_state_dict']['token_embedding_table.weight'].shape[0]

    if saved_vocab_size == len(base_chars):
        print("-> Base model detected. Loading weights and expanding vocabulary matrices to include special tokens...")
        model.load_state_dict(checkpoint['model_state_dict'])
        model.resize_token_embeddings(len(all_tokens))
    elif saved_vocab_size == len(all_tokens):
        print("-> Resized checkpoint detected. Aligning internal architecture layers prior to weight loading...")
        model.resize_token_embeddings(len(all_tokens))
        model.load_state_dict(checkpoint['model_state_dict'])
    else:
        print(f"-> Warning: Size mismatch (Saved: {saved_vocab_size}, Target: {len(all_tokens)}). Forcing architecture rebuild...")
        model.resize_token_embeddings(len(all_tokens))
else:
    print("No previous checkpoint detected. Initializing a fresh model layout from scratch...")
    model.resize_token_embeddings(len(all_tokens))

# ===================================================
# 4. FINE-TUNING STRATEGY LOOP
# ===================================================
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5)
epochs = 150
batch_size_ft = 2

print("\nExecuting comprehensive chat fine-tuning parameters...")
model.train()

for epoch in range(epochs):
    permutation = torch.randperm(X_train.size(0))
    epoch_loss = 0

    for i in range(0, X_train.size(0), batch_size_ft):
        optimizer.zero_grad(set_to_none=True)

        indices = permutation[i:i+batch_size_ft]
        batch_x, batch_y = X_train[indices].to(device), Y_train[indices].to(device)

        logits, loss = model(batch_x, batch_y)

        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    if (epoch + 1) % 30 == 0:
        print(f"Epoch {epoch+1}/{epochs} | Production Loss: {epoch_loss / (X_train.size(0)/batch_size_ft):.4f}")

# Save updated fine-tuned model checkpoint
print("\nExporting advanced fine-tuned structural layers...")
fine_tuned_checkpoint = {
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'vocab_size': len(all_tokens),
    'chars': all_tokens
}
torch.save(fine_tuned_checkpoint, model_filename)
print(f"Checkpoint safely updated inside '{model_filename}'.")

# ===================================================
# 5. CONVERSATIONAL TESTING & INFERENCE
# ===================================================
model.eval()
print("\n--- Testing Production Structural Prompting ---")

def chat_inference(system_prompt, user_query):
    # Assemble the input sequence using the exact structural tags the model was trained on
    prompt = f"[BOS][SYSTEM]{system_prompt}[USER]{user_query}[THOUGHT]"

    encoded = [stoi[c] for c in prompt if c in stoi]
    input_tensor = torch.tensor([encoded], dtype=torch.long, device=device)

    # Set generation length to accommodate both the internal reasoning and the final response
    output_ids = model.generate(input_tensor, max_new_tokens=60).tolist()[0]

    # Extract only the newly generated tokens
    new_tokens = output_ids[len(encoded):]

    # Decode integers back to text
    decoded_output = "".join([itos[idx] for idx in new_tokens if idx != PAD_ID])
    return decoded_output

# Test the model and watch it output its internal reasoning chain before answering
print(chat_inference("You are a helpful math assistant.", "What is 2+2?"))

Found existing checkpoint 'mini_llm_model.pt'. Validating structures...
-> Warning: Size mismatch (Saved: 65, Target: 81). Forcing architecture rebuild...


AttributeError: 'MiniLanguageModel' object has no attribute 'resize_token_embeddings'

In [ ]:
!nvidia-smi

Tue Jul 21 22:01:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   57C    P0             28W /   70W |    2833MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----